In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/atomicd/retail-store-inventory-and-demand-forecasting/sales_data.csv


In [10]:
from pathlib import Path
import pandas as pd

csv_files = list(Path("/kaggle/input").rglob("*.csv"))

print("CSV files found:", len(csv_files))

for file in csv_files:
    print(file)

from pathlib import Path

csv_files = list(Path("/kaggle/input").rglob("*.csv"))

for file in csv_files:
    print(file)

CSV files found: 1
/kaggle/input/datasets/atomicd/retail-store-inventory-and-demand-forecasting/sales_data.csv
/kaggle/input/datasets/atomicd/retail-store-inventory-and-demand-forecasting/sales_data.csv


In [11]:
import pandas as pd

# Load the dataset
file_path = "/kaggle/input/datasets/atomicd/retail-store-inventory-and-demand-forecasting/sales_data.csv"

df = pd.read_csv(file_path)

# 1. Dataset dimensions
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

# 2. Actual column names
print("\nCOLUMN NAMES:")
print(df.columns.tolist())

# 3. First five rows
print("\nFIRST FIVE ROWS:")
print(df.head())

# 4. Data types
print("\nDATA TYPES:")
print(df.dtypes)

# 5. Missing values
print("\nMISSING VALUES:")
print(df.isnull().sum())

# 6. Duplicate rows
print("\nDUPLICATE ROWS:")
print(df.duplicated().sum())

# 7. Statistical summary
print("\nSTATISTICAL SUMMARY:")
print(df.describe(include="all").T)

DATASET OVERVIEW
Total rows: 76000
Total columns: 16

COLUMN NAMES:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality', 'Epidemic', 'Demand']

FIRST FIVE ROWS:
         Date Store ID Product ID     Category Region  Inventory Level  \
0  2022-01-01     S001      P0001  Electronics  North              195   
1  2022-01-01     S001      P0002     Clothing  North              117   
2  2022-01-01     S001      P0003     Clothing  North              247   
3  2022-01-01     S001      P0004  Electronics  North              139   
4  2022-01-01     S001      P0005    Groceries  North              152   

   Units Sold  Units Ordered  Price  Discount Weather Condition  Promotion  \
0         102            252  72.72         5             Snowy          0   
1         117            249  80.16        15             Snowy          1   
2         114 

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================

# 1. LOAD DATASET

# ==========================================

file_path = (
"/kaggle/input/datasets/atomicd/"
"retail-store-inventory-and-demand-forecasting/"
"sales_data.csv"
)

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Original shape:", df.shape)

# ==========================================

# 2. STANDARDIZE COLUMN NAMES

# ==========================================

df.columns = (
df.columns.astype(str)
.str.strip()
.str.lower()
.str.replace(r"\s+", "_", regex=True)
)

print("\nStandardized columns:")
print(df.columns.tolist())

# ==========================================

# 3. VALIDATE REQUIRED COLUMNS

# ==========================================

required_columns = [
"date",
"store_id",
"product_id",
"price",
"units_sold",
]

missing_columns = [
col for col in required_columns
if col not in df.columns
]

if missing_columns:
    raise ValueError(
    f"Required columns missing: {missing_columns}"
    )

# ==========================================

# 4. CONVERT DATA TYPES

# ==========================================

df["date"] = pd.to_datetime(
df["date"],
errors="coerce"
)

numeric_columns = [
"price",
"units_sold",
"discount",
"promotion",
"competitor_pricing",
"inventory_level",
"epidemic",
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
        )

# ==========================================

# 5. CHECK DATA QUALITY

# ==========================================

print("\nMissing values before cleaning:")
print(df.isnull().sum())

print("\nDuplicate rows before cleaning:")
print(df.duplicated().sum())

# Remove exact duplicate records

df = df.drop_duplicates().copy()

# Remove records missing essential variables

df = df.dropna(
subset=[
"date",
"store_id",
"product_id",
"price",
"units_sold",
]
).copy()

# Remove impossible or invalid treatment/outcome values

df = df[
(df["price"] > 0)
& (df["units_sold"] >= 0)
].copy()

print("\nShape after basic cleaning:", df.shape)

# ==========================================

# 6. CREATE TIME FEATURES

# ==========================================

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek

# ==========================================

# 7. CHECK PRICE VARIATION

# ==========================================

print("\nOverall price statistics:")
print(df["price"].describe())

print("\nNumber of unique prices:", df["price"].nunique())

print("\nProducts with price variation:")
price_variation = (
df.groupby(["store_id", "product_id"])["price"]
.nunique()
)

print("Store-product combinations:", len(price_variation))
print("Combinations with varying prices:",
(price_variation > 1).sum())

# ==========================================

# 8. SELECT CANDIDATE DML VARIABLES

# ==========================================

treatment = "price"
outcome = "units_sold"

candidate_features = [
"discount",
"promotion",
"competitor_pricing",
"inventory_level",
"weather_condition",
"seasonality",
"epidemic",
"product_id",
"store_id",
"category",
"region",
"year",
"month",
"day_of_week",
]

features = [
col for col in candidate_features
if col in df.columns
]

print("\nTreatment:", treatment)
print("Outcome:", outcome)
print("Candidate control variables:", features)

# ==========================================

# 9. SAVE CLEANED DATA

# ==========================================

output_path = "/kaggle/working/cleaned_sales_data.csv"

df.to_csv(output_path, index=False)

print("\nCleaned dataset saved to:", output_path)
print("Final shape:", df.shape)

print("\nData processing completed.")


Dataset loaded successfully.
Original shape: (76000, 16)

Standardized columns:
['date', 'store_id', 'product_id', 'category', 'region', 'inventory_level', 'units_sold', 'units_ordered', 'price', 'discount', 'weather_condition', 'promotion', 'competitor_pricing', 'seasonality', 'epidemic', 'demand']

Missing values before cleaning:
date                  0
store_id              0
product_id            0
category              0
region                0
inventory_level       0
units_sold            0
units_ordered         0
price                 0
discount              0
weather_condition     0
promotion             0
competitor_pricing    0
seasonality           0
epidemic              0
demand                0
dtype: int64

Duplicate rows before cleaning:
0

Shape after basic cleaning: (76000, 16)

Overall price statistics:
count    76000.000000
mean        67.726028
std         39.377899
min          4.740000
25%         31.997500
50%         64.500000
75%         95.830000
max        2